In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import sph_harm_y, genlaguerre
from math import factorial

def radial_wavefunction(n, l, r, a0=1.0):
    """
    Calculates the radial part of the hydrogen wavefunction R_{nl}(r).
    """
    rho = (2 * r) / (n * a0)
    norm = np.sqrt((2 / (n * a0))**3 * factorial(n - l - 1) / (2 * n * factorial(n + l)))
    return norm * np.exp(-rho / 2) * (rho**l) * genlaguerre(n - l - 1, 2 * l + 1)(rho)

# Set white background style and serif fonts globally
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif', 'serif']
plt.rcParams['font.family'] = 'serif'
plt.rcParams['mathtext.fontset'] = 'cm'
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'

def hydrogen_wavefunction(n, l, m, r, theta, phi, a0=1.0):
    """
    Calculates the full hydrogen wavefunction psi_{nlm}(r, theta, phi).
    """
    R = radial_wavefunction(n, l, r, a0)
    # scipy.special.sph_harm_y takes (l, m, theta, phi)
    # Standard physics convention: theta = polar [0, pi], phi = azimuthal [0, 2pi]
    Y = sph_harm_y(l, m, theta, phi)
    return R * Y

def plot_orbital_contour(n, l, m, extent=15, res=200):
    """
    Plots the probability density contour for a given state in the xz-plane.
    """
    # Create a grid in the xz-plane (y=0)
    x = np.linspace(-extent, extent, res)
    z = np.linspace(-extent, extent, res)
    X, Z = np.meshgrid(x, z)

    # Convert Cartesian (x, 0, z) to Spherical (r, theta, phi)
    r = np.sqrt(X**2 + Z**2)
    theta = np.arccos(Z / (r + 1e-15))  # Polar angle (0 to pi)
    phi = np.arctan2(0, X)             # Azimuthal angle

    # Calculate wavefunction
    psi = hydrogen_wavefunction(n, l, m, r, theta, phi)
    prob_density = np.abs(psi)**2

    # Plotting
    plt.figure(figsize=(7, 6))
    # Use 'Blues' colormap so background (zero probability) is white
    contour = plt.contourf(X, Z, prob_density, levels=50, cmap='Blues')
    plt.colorbar(contour, label=r'Probability Density $|\psi|^2$')
    plt.title(rf'Hydrogen Orbital Probability Density: $n={n}, \ell={l}, m_\ell={m}$')
    plt.xlabel('$x / a_0$')
    plt.ylabel('$z / a_0$')
    plt.axis('equal')
    plt.show()

In [ ]:
# Plotting a few representative orbitals
# 2p_z orbital (n=2, l=1, m=0)
plot_orbital_contour(n=2, l=1, m=0)

# 3d_{z^2} orbital (n=3, l=2, m=0)
plot_orbital_contour(n=3, l=2, m=0, extent=25)

# 3d orbital (n=3, l=2, m=1)
plot_orbital_contour(n=3, l=2, m=1, extent=25)

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
from skimage import measure

def plot_orbital_3d(n, l, m, extent=15, res=60, level_frac=0.1):
    """
    Plots a 3D isosurface of the probability density using marching cubes.
    """
    # Create a 3D grid
    coords = np.linspace(-extent, extent, res)
    X, Y, Z = np.meshgrid(coords, coords, coords)

    # Convert Cartesian to Spherical
    r = np.sqrt(X**2 + Y**2 + Z**2)
    theta = np.arccos(Z / (r + 1e-15))
    phi = np.arctan2(Y, X)

    # Calculate probability density
    psi = hydrogen_wavefunction(n, l, m, r, theta, phi)
    prob = np.abs(psi)**2

    # Set the isosurface level
    max_prob = np.max(prob)
    level = level_frac * max_prob

    print(rf"Orbital (n={n}, l={l}, m={m})")
    print(f"  Max probability density: {max_prob:.6e}")
    print(f"  Isosurface level ({level_frac*100}% of max): {level:.6e}")

    try:
        # Generate the surface mesh
        verts, faces, normals, values = measure.marching_cubes(prob, level)

        # Map index-coordinates back to the physical scale
        step = 2 * extent / (res - 1)
        verts = verts * step - extent

        fig = plt.figure(figsize=(10, 8), facecolor='white')
        ax = fig.add_subplot(111, projection='3d')

        # Plot the surface
        mesh = ax.plot_trisurf(verts[:, 0], verts[:, 1], faces, verts[:, 2],
                               cmap='Blues_r', lw=1, alpha=0.7, edgecolor='none')

        # Fixed LaTeX titles using standard commands
        ax.set_title(rf'3D Isosurface ($|\psi|^2$): $n={n}, \ell={l}, m_\ell={m}$', fontsize=20)
        ax.set_xlabel('$x/a_0$', fontsize=14)
        ax.set_ylabel('$y/a_0$', fontsize=14)
        ax.set_zlabel('$z/a_0$', fontsize=14)

        ax.set_box_aspect([1,1,1])
        plt.show()

    except ValueError:
        print(f"No surface found at level {level:.6e}. Try lowering level_frac.")

# Visualize with the updated function
plot_orbital_3d(n=2, l=1, m=0)
plot_orbital_3d(n=3, l=2, m=0, extent=25)

In [ ]:
def plot_orbitals_grid_3d(states_list, extent=25, res=50, level_frac=0.1):
    """
    Plots a grid of 3D orbital isosurfaces for comparison.
    """
    n_plots = len(states_list)
    fig = plt.figure(figsize=(10, 7 * n_plots), facecolor='white')

    for i, (n, l, m) in enumerate(states_list):
        ax = fig.add_subplot(n_plots, 1, i + 1, projection='3d')

        # Create grid for this specific subplot
        coords = np.linspace(-extent, extent, res)
        X, Y, Z = np.meshgrid(coords, coords, coords)
        r = np.sqrt(X**2 + Y**2 + Z**2)
        theta = np.arccos(Z / (r + 1e-15))
        phi = np.arctan2(Y, X)

        # Calculate probability density
        psi = hydrogen_wavefunction(n, l, m, r, theta, phi)
        prob = np.abs(psi)**2

        max_prob = np.max(prob)
        level = level_frac * max_prob

        print(rf"Subplot (n={n}, l={l}, m={m}) | Isosurface Level: {level:.6e}")

        try:
            # Generate mesh
            verts, faces, normals, values = measure.marching_cubes(prob, level)
            step = 2 * extent / (res - 1)
            verts = verts * step - extent

            # Plot trisurf
            ax.plot_trisurf(verts[:, 0], verts[:, 1], faces, verts[:, 2],
                           cmap='Blues_r', lw=0.5, alpha=0.7, edgecolor='none')

            # Corrected title with standard LaTeX symbols
            ax.set_title(rf'3D Orbital: $n={n}, \ell={l}, m_\ell={m}$', fontsize=18, pad=20)
            ax.set_xlabel('$x/a_0$')
            ax.set_ylabel('$y/a_0$')
            ax.set_zlabel('$z/a_0$')
            ax.set_box_aspect([1, 1, 1])

        except ValueError:
            ax.text2D(0.5, 0.5, f"No surface at {level_frac*100}%", transform=ax.transAxes, ha='center')

    plt.tight_layout()
    plt.show()

# Define the 1s, 2p, and 3d states (using m=0 for z-aligned visualization)
comparison_states = [
    (1, 0, 0),  # 1s
    (2, 1, 0),  # 2p_z
    (3, 2, 0)   # 3d_z^2
]

plot_orbitals_grid_3d(comparison_states, extent=20)

In [ ]:
# Define the quantum states for the visualization grid
row_1s = [(1, 0, 0)]
row_2p = [(2, 1, -1), (2, 1, 0), (2, 1, 1)]
row_3d = [(3, 2, -2), (3, 2, -1), (3, 2, 0), (3, 2, 1), (3, 2, 2)]

print(f"1s states: {row_1s}")
print(f"2p states: {row_2p}")
print(f"3d states: {row_3d}")

In [ ]:
from matplotlib.ticker import MaxNLocator

def plot_complex_orbitals_3x5(states_1s, states_2p, states_3d, res=60, level_frac=0.15,
                             row_limits=[(-1.5, 1.5), (-7, 7), (-15, 15)]):
    """
    Visualizes complex hydrogen orbitals in a 3x5 grid using simplified labels (1s, 2p, 3d).
    """
    fig = plt.figure(figsize=(24, 18), facecolor='white')

    # Mapping for labels based on l
    label_map = {0: '1s', 1: '2p', 2: '3d'}

    # Mapping of states to specific subplot indices (1-indexed for subplot)
    mapping = [
        (states_1s, [3], 0),
        (states_2p, [7, 8, 9], 1),
        (states_3d, [11, 12, 13, 14, 15], 2)
    ]

    for states, indices, row_idx in mapping:
        rmin, rmax = row_limits[row_idx]
        for (n, l, m), idx in zip(states, indices):
            ax = fig.add_subplot(3, 5, idx, projection='3d')

            coords = np.linspace(rmin, rmax, res)
            X, Y, Z = np.meshgrid(coords, coords, coords)

            r = np.sqrt(X**2 + Y**2 + Z**2)
            theta = np.arccos(Z / (r + 1e-15))
            phi = np.arctan2(Y, X)

            psi = hydrogen_wavefunction(n, l, m, r, theta, phi)
            prob = np.abs(psi)**2

            max_p = np.max(prob)
            level = level_frac * max_p

            try:
                verts, faces, normals, values = measure.marching_cubes(prob, level)
                step = (rmax - rmin) / (res - 1)
                verts = verts * step + rmin

                ax.plot_trisurf(verts[:, 0], verts[:, 1], faces, verts[:, 2],
                                cmap='Oranges_r', lw=0, alpha=0.9, edgecolor='none')

                ax.set_xlabel('$x/a_0$', fontsize=20)
                ax.set_ylabel('$y/a_0$', fontsize=20)
                ax.set_zlabel('$z/a_0$', fontsize=20)
                ax.set_box_aspect([1, 1, 1])

                ax.tick_params(axis='both', which='major', labelsize=14)
                ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
                ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
                ax.zaxis.set_major_locator(MaxNLocator(nbins=3))

                # Replaced n, l, m string with simple label
                label_text = rf'${label_map[l]}$'
                ax.text2D(0.5, -0.15, label_text, transform=ax.transAxes,
                          ha='center', va='top', fontsize=32, fontweight='bold')

                ax.set_xlim(rmin, rmax)
                ax.set_ylim(rmin, rmax)
                ax.set_zlim(rmin, rmax)

            except ValueError:
                ax.text2D(0.5, 0.5, "Surface error", transform=ax.transAxes, ha='center')

    plt.tight_layout(rect=[0, 0.05, 0.92, 0.95], h_pad=3.0, w_pad=4.0)
    plt.show()

# Execute the 3D complex grid
plot_complex_orbitals_3x5(row_1s, row_2p, row_3d)

In [ ]:
def plot_complex_contours_3x5(states_1s, states_2p, states_3d, res=150,
                             row_limits=[(-1.5, 1.5), (-7, 7), (-15, 15)]):
    """
    Visualizes 2D probability density contours for complex orbitals with simplified labels (1s, 2p, 3d).
    """
    fig = plt.figure(figsize=(24, 18), facecolor='white')

    # Mapping for labels based on l
    label_map = {0: '1s', 1: '2p', 2: '3d'}

    mapping = [
        (states_1s, [3], 0),
        (states_2p, [7, 8, 9], 1),
        (states_3d, [11, 12, 13, 14, 15], 2)
    ]

    for states, indices, row_idx in mapping:
        rmin, rmax = row_limits[row_idx]
        for (n, l, m), idx in zip(states, indices):
            ax = fig.add_subplot(3, 5, idx)

            coords = np.linspace(rmin, rmax, res)
            X, Z = np.meshgrid(coords, coords)

            r = np.sqrt(X**2 + Z**2)
            theta = np.arccos(Z / (r + 1e-15))
            phi = np.arctan2(0, X)

            psi = hydrogen_wavefunction(n, l, m, r, theta, phi)
            prob = np.abs(psi)**2

            contour = ax.contourf(X, Z, prob, levels=50, cmap='Oranges')

            ax.set_xlabel('$x/a_0$', fontsize=20)
            ax.set_ylabel('$z/a_0$', fontsize=20)
            ax.tick_params(axis='both', labelsize=16)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.set_aspect('equal')

            # Replaced n, l, m string with simple label
            label_text = rf'${label_map[l]}$'
            ax.text(0.5, -0.25, label_text, transform=ax.transAxes,
                    ha='center', va='top', fontsize=32, fontweight='bold')

    plt.tight_layout(rect=[0, 0.05, 0.95, 0.95], h_pad=3.0, w_pad=6.0)
    plt.show()

# Execute the 2D complex grid
plot_complex_contours_3x5(row_1s, row_2p, row_3d)

In [ ]:
def real_hydrogen_wavefunction(n, l, m, r, theta, phi, a0=1.0):
    """
    Calculates the real hydrogen wavefunction psi_{nlm}(r, theta, phi).

    Args:
        n, l, m (int): Quantum numbers
        r, theta, phi (array): Spherical coordinates
        a0 (float): Bohr radius

    Returns:
        psi_real (array): The real-valued wavefunction
    """
    if m == 0:
        # For m=0, the wavefunction is already real
        psi = hydrogen_wavefunction(n, l, 0, r, theta, phi, a0)
        return psi.real
    elif m > 0:
        # Real part of the complex wavefunction scaled by sqrt(2)
        psi = hydrogen_wavefunction(n, l, m, r, theta, phi, a0)
        return np.sqrt(2) * psi.real
    else:  # m < 0
        # Imaginary part of the complex wavefunction for |m| scaled by sqrt(2)
        psi = hydrogen_wavefunction(n, l, abs(m), r, theta, phi, a0)
        return np.sqrt(2) * psi.imag

# Test the function with the 2p_x orbital (n=2, l=1, m=1)
r_test = 2.0
theta_test = np.pi/2
phi_test = 0.0

psi_2px = real_hydrogen_wavefunction(2, 1, 1, r_test, theta_test, phi_test)
print(f"Value of 2p_x wavefunction at r=2, theta=pi/2, phi=0: {psi_2px:.6f}")
print(f"Data type of result: {type(psi_2px)}")

In [ ]:
# Define the real orbital states for the visualization grid
real_row_1s = [(1, 0, 0, '1s')]
real_row_2p = [
    (2, 1, 1, '2p_x'),
    (2, 1, -1, '2p_y'),
    (2, 1, 0, '2p_z')
]
real_row_3d = [
    (3, 2, -2, '3d_{xy}'),
    (3, 2, -1, '3d_{yz}'),
    (3, 2, 0, '3d_{z^2}'),
    (3, 2, 1, '3d_{xz}'),
    (3, 2, 2, '3d_{x^2-y^2}')
]

print(f"Row 1 (1s): {real_row_1s}")
print(f"Row 2 (2p): {real_row_2p}")
print(f"Row 3 (3d): {real_row_3d}")

In [ ]:
from matplotlib.ticker import MaxNLocator

def plot_real_orbitals_3x5(states_1s, states_2p, states_3d, res=60, level_frac=0.15,
                           row_limits=[(-6, 6), (-13, 13), (-25, 25)]):
    """
    Visualizes 'real' hydrogen orbitals in a 3x5 grid with custom row limits.
    row_limits: List of (min, max) tuples for the axes of each row.
    """
    fig = plt.figure(figsize=(24, 18), facecolor='white')

    # Mapping of states, subplot indices, and their corresponding row limit index
    mapping = [
        (states_1s, [3], 0),
        (states_2p, [7, 8, 9], 1),
        (states_3d, [11, 12, 13, 14, 15], 2)
    ]

    for states, indices, row_idx in mapping:
        rmin, rmax = row_limits[row_idx]
        for (n, l, m, label), idx in zip(states, indices):
            ax = fig.add_subplot(3, 5, idx, projection='3d')

            # Create a 3D grid using the specified row limits
            coords = np.linspace(rmin, rmax, res)
            X, Y, Z = np.meshgrid(coords, coords, coords)

            r = np.sqrt(X**2 + Y**2 + Z**2)
            theta = np.arccos(Z / (r + 1e-15))
            phi = np.arctan2(Y, X)

            psi = real_hydrogen_wavefunction(n, l, m, r, theta, phi)
            prob = psi**2

            max_p = np.max(prob)
            level = level_frac * max_p

            try:
                verts, faces, normals, values = measure.marching_cubes(prob, level)
                # Scale vertices from index-space to physical coordinate-space
                step = (rmax - rmin) / (res - 1)
                verts = verts * step + rmin

                # Render the 3D surface
                ax.plot_trisurf(verts[:, 0], verts[:, 1], faces, verts[:, 2],
                                cmap='Blues_r', lw=0, alpha=0.9, edgecolor='none')

                # Show axes and labels
                ax.set_xlabel('$x$', fontsize=20)
                ax.set_ylabel('$y$', fontsize=20)
                ax.set_zlabel('$z$', fontsize=20)
                ax.set_box_aspect([1, 1, 1])

                # Increase tick label size and reduce number of ticks
                ax.tick_params(axis='both', which='major', labelsize=14)
                ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
                ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
                ax.zaxis.set_major_locator(MaxNLocator(nbins=3))

                # Orbital label positioned further underneath
                ax.text2D(0.5, -0.15, rf'${label}$', transform=ax.transAxes,
                          ha='center', va='top', fontsize=32, fontweight='bold')

                # Apply the specific limits for this row
                ax.set_xlim(rmin, rmax)
                ax.set_ylim(rmin, rmax)
                ax.set_zlim(rmin, rmax)

            except ValueError:
                ax.text2D(0.5, 0.5, "Surface error", transform=ax.transAxes, ha='center')

    # Reduced h_pad to bring rows closer together
    plt.tight_layout(rect=[0, 0.05, 0.92, 0.95], h_pad=3.0, w_pad=6.0)
    plt.show()

# Using the tighter limits from your last request
custom_limits = [(-1.5, 1.5), (-7, 7), (-15, 15)]

# Execute with custom limits
plot_real_orbitals_3x5(real_row_1s, real_row_2p, real_row_3d, row_limits=custom_limits)

In [ ]:
def plot_real_contours_3x5(states_1s, states_2p, states_3d, res=150,
                           row_limits=[(-1.5, 1.5), (-7, 7), (-15, 15)]):
    """
    Visualizes 2D probability density contours in the optimal plane for each orbital.
    """
    fig = plt.figure(figsize=(24, 18), facecolor='white')

    # Plane mapping to show the characteristic lobes of each orbital
    plane_map = {
        '1s': 'xz', '2p_x': 'xz', '2p_y': 'yz', '2p_z': 'xz',
        '3d_{xy}': 'xy', '3d_{yz}': 'yz', '3d_{z^2}': 'xz',
        '3d_{xz}': 'xz', '3d_{x^2-y^2}': 'xy'
    }

    mapping = [
        (states_1s, [3], 0),
        (states_2p, [7, 8, 9], 1),
        (states_3d, [11, 12, 13, 14, 15], 2)
    ]

    for states, indices, row_idx in mapping:
        rmin, rmax = row_limits[row_idx]
        for (n, l, m, label), idx in zip(states, indices):
            ax = fig.add_subplot(3, 5, idx)

            # Choose plane and assign coordinates for the grid
            plane = plane_map.get(label, 'xz')
            coords = np.linspace(rmin, rmax, res)
            C1, C2 = np.meshgrid(coords, coords)

            if plane == 'xz':
                X, Y, Z = C1, 0, C2
                xlab, ylab = '$x/a_0$', '$z/a_0$'
            elif plane == 'yz':
                X, Y, Z = 0, C1, C2
                xlab, ylab = '$y/a_0$', '$z/a_0$'
            elif plane == 'xy':
                X, Y, Z = C1, C2, 0
                xlab, ylab = '$x/a_0$', '$y/a_0$'

            # Convert selected plane coordinates to Spherical
            r = np.sqrt(X**2 + Y**2 + Z**2)
            theta = np.arccos(Z / (r + 1e-15))
            phi = np.arctan2(Y, X)

            # Calculate probability density
            psi = real_hydrogen_wavefunction(n, l, m, r, theta, phi)
            prob = psi**2

            # Plot contours using the local grid coordinates
            contour = ax.contourf(C1, C2, prob, levels=50, cmap='Blues')

            # Style axes with relevant labels
            ax.set_xlabel(xlab, fontsize=20)
            ax.set_ylabel(ylab, fontsize=20)
            ax.tick_params(axis='both', labelsize=16)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.set_aspect('equal')

            # Orbital label underneath
            ax.text(0.5, -0.25, rf'${label}$', transform=ax.transAxes,
                    ha='center', va='top', fontsize=32, fontweight='bold')

    # Reduced h_pad to reduce the space between rows
    plt.tight_layout(rect=[0, 0.05, 0.95, 0.95], h_pad=3.0, w_pad=6.0)
    plt.show()

# Execute 2D contour grid with optimized slices
plot_real_contours_3x5(real_row_1s, real_row_2p, real_row_3d)

In [ ]:
# Define target real orbital states
orbital_1s = (1, 0, 0, '1s')

orbitals_2p = [
    (2, 1, 1, '2p_x'),
    (2, 1, -1, '2p_y'),
    (2, 1, 0, '2p_z')
]

orbitals_3d = [
    (3, 2, -2, '3d_{xy}'),
    (3, 2, -1, '3d_{yz}'),
    (3, 2, 0, '3d_{z^2}'),
    (3, 2, 1, '3d_{xz}'),
    (3, 2, 2, '3d_{x^2-y^2}')
]

print("1s state:", orbital_1s)
print("2p real orbitals:", [o[3] for o in orbitals_2p])
print("3d real orbitals:", [o[3] for o in orbitals_3d])

In [ ]:
from scipy.special import sph_harm_y

def hydrogen_wavefunction(n, l, m, r, theta, phi, a0=1.0):
    """
    Calculates the full hydrogen wavefunction psi_{nlm}(r, theta, phi)
    using the non-deprecated sph_harm_y function.
    """
    R = radial_wavefunction(n, l, r, a0)
    # sph_harm_y(l, m, theta, phi) where theta is polar and phi is azimuthal
    Y = sph_harm_y(l, m, theta, phi)
    return R * Y

def plot_orbital_cloud(ax, n, l, m, label, extent=15, res=55, threshold=0.005):
    """
    Visualizes an orbital as a 3D volumetric cloud with significantly enhanced visibility.
    """
    # 1. Generate 3D Cartesian grid with higher resolution
    coords = np.linspace(-extent, extent, res)
    X, Y, Z = np.meshgrid(coords, coords, coords)

    # 2. Convert to spherical coordinates
    r = np.sqrt(X**2 + Y**2 + Z**2)
    theta = np.arccos(Z / (r + 1e-15))
    phi = np.arctan2(Y, X)

    # 3. Compute real probability density
    psi = real_hydrogen_wavefunction(n, l, m, r, theta, phi)
    prob = np.abs(psi)**2

    # 4. Normalize and filter low-probability points
    max_p = np.max(prob)
    prob_norm = prob / max_p if max_p > 0 else prob

    # Lower threshold to include more of the 'fringe' of the cloud
    mask = prob_norm > threshold

    x_f = X[mask].flatten()
    y_f = Y[mask].flatten()
    z_f = Z[mask].flatten()
    p_f = prob_norm[mask].flatten()

    # 5. Map density to color and transparency
    # Use a darker part of the Blues map for better visibility
    colors = plt.cm.Blues(0.4 + 0.6 * p_f)

    # Use power scaling (p^0.4) to 'boost' the visibility of lower density areas
    colors[:, -1] = np.clip(0.7 * p_f**0.4, 0, 1)

    # 6. Render scatter plot with larger markers
    ax.scatter(x_f, y_f, z_f, facecolors=colors, s=12, edgecolors='none')

    # 7. Standardize visual output
    ax.set_title(rf'Cloud Model: ${label}$', fontsize=18)
    ax.set_xlabel('$x$', fontsize=24)
    ax.set_ylabel('$y$', fontsize=24)
    ax.set_zlabel('$z$', fontsize=24)
    ax.set_box_aspect([1, 1, 1])
    ax.set_xlim(-extent, extent)
    ax.set_ylim(-extent, extent)
    ax.set_zlim(-extent, extent)


In [ ]:
from matplotlib.ticker import MaxNLocator

def plot_real_clouds_3x5(states_1s, states_2p, states_3d, res=50, threshold=0.005,
                          row_extents=[6, 11, 18]):
    """
    Visualizes 'real' hydrogen orbitals in a 3x5 grid using the Volumetric Cloud model.
    """
    fig = plt.figure(figsize=(24, 18), facecolor='white')

    # Mapping of states, subplot indices, and their corresponding row extent index
    mapping = [
        (states_1s, [3], 0),
        (states_2p, [7, 8, 9], 1),
        (states_3d, [11, 12, 13, 14, 15], 2)
    ]

    for states, indices, row_idx in mapping:
        extent = row_extents[row_idx]
        for (n, l, m, label), idx in zip(states, indices):
            ax = fig.add_subplot(3, 5, idx, projection='3d')

            # Call the volumetric cloud function
            plot_orbital_cloud(ax, n, l, m, label, extent=extent, res=res, threshold=threshold)

            # Clear the default title set by plot_orbital_cloud to move the label underneath
            ax.set_title('')

            # Refine styling for the 3x5 layout
            ax.tick_params(axis='both', which='major', labelsize=12)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.zaxis.set_major_locator(MaxNLocator(nbins=3))

            # Position the label underneath the plot and set fontsize to 32
            ax.text2D(0.5, -0.15, rf'${label}$', transform=ax.transAxes,
                      ha='center', va='top', fontsize=32, fontweight='bold')

    # Tighten layout and display
    plt.tight_layout(rect=[0, 0.05, 0.95, 0.95], h_pad=4.0, w_pad=3.0)
    plt.show()

# Execute the refined 3x5 cloud grid visualization
plot_real_clouds_3x5(real_row_1s, real_row_2p, real_row_3d)

In [ ]:
from skimage import measure

def plot_orbital_isosurface(ax, n, l, m, label, extent=15, res=50, level_frac=0.3):
    """
    Visualizes an orbital as a solid 3D isosurface using the marching cubes algorithm.
    """
    # 1. Generate 3D Cartesian grid
    coords = np.linspace(-extent, extent, res)
    X, Y, Z = np.meshgrid(coords, coords, coords)

    # 2. Convert to spherical coordinates
    r = np.sqrt(X**2 + Y**2 + Z**2)
    theta = np.arccos(Z / (r + 1e-15))
    phi = np.arctan2(Y, X)

    # 3. Calculate probability density |psi|^2
    psi = real_hydrogen_wavefunction(n, l, m, r, theta, phi)
    prob = np.abs(psi)**2

    # 4. Determine isosurface level
    max_p = np.max(prob)
    level = level_frac * max_p

    try:
        # 5. Generate surface mesh
        verts, faces, normals, values = measure.marching_cubes(prob, level)

        # 6. Map mesh indices back to coordinate space
        step = (2 * extent) / (res - 1)
        verts = verts * step - extent

        # 7. Render surface
        ax.plot_trisurf(verts[:, 0], verts[:, 1], faces, verts[:, 2],
                        cmap='Blues_r', lw=0, alpha=0.8, edgecolor='none')

        # 8. Standardize labels and title
        ax.set_title(rf'Isosurface: ${label}$', fontsize=18)
        ax.set_xlabel('$x$', fontsize=24)
        ax.set_ylabel('$y$', fontsize=24)
        ax.set_zlabel('$z$', fontsize=24)
        ax.set_box_aspect([1, 1, 1])
        ax.set_xlim(-extent, extent)
        ax.set_ylim(-extent, extent)
        ax.set_zlim(-extent, extent)

    except ValueError:
        ax.text2D(0.5, 0.5, "No surface found", transform=ax.transAxes, ha='center')


In [ ]:
from matplotlib.ticker import MaxNLocator

def plot_real_isosurfaces_3x5(states_1s, states_2p, states_3d, res=50, level_frac=0.05,
                               row_extents=[6, 11, 18]):
    """
    Visualizes 'real' hydrogen orbitals in a 3x5 grid using the Solid Isosurface model.
    """
    fig = plt.figure(figsize=(24, 18), facecolor='white')

    # Mapping of states, subplot indices, and their corresponding row extent index
    mapping = [
        (states_1s, [3], 0),
        (states_2p, [7, 8, 9], 1),
        (states_3d, [11, 12, 13, 14, 15], 2)
    ]

    for states, indices, row_idx in mapping:
        extent = row_extents[row_idx]
        for (n, l, m, label), idx in zip(states, indices):
            ax = fig.add_subplot(3, 5, idx, projection='3d')

            # Call the isosurface function
            plot_orbital_isosurface(ax, n, l, m, label, extent=extent, res=res, level_frac=level_frac)

            # Clear the default title set by plot_orbital_isosurface to move the label underneath
            ax.set_title('')

            # Refine styling for the 3x5 layout
            ax.tick_params(axis='both', which='major', labelsize=12)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.zaxis.set_major_locator(MaxNLocator(nbins=3))

            # Position the label underneath the plot and set fontsize to 32
            ax.text2D(0.5, -0.15, rf'${label}$', transform=ax.transAxes,
                      ha='center', va='top', fontsize=32, fontweight='bold')


    # Tighten layout and display
    plt.tight_layout(rect=[0, 0.05, 0.95, 0.95], h_pad=4.0, w_pad=3.0)
    plt.show()

# Execute the 3x5 isosurface grid visualization
plot_real_isosurfaces_3x5(real_row_1s, real_row_2p, real_row_3d)

# Task
Implement a function to calculate the specific probability density threshold required to enclose a target fraction (e.g., 90%) of the total probability within a 3D volume. Update the 3D isosurface visualization function to use this enclosure threshold instead of a fixed percentage of the maximum density, then regenerate the 3x5 grid of real hydrogen orbitals (1s, 2p, 3d) to show surfaces containing 90% of the electron probability. Finally, summarize how this 90% enclosure method alters the scale and appearance of the orbitals compared to the previous fixed-percentage visualization.

## Implement Probability Threshold Function

### Subtask:
Develop a function to determine the probability density value that encloses a specific fraction (e.g., 90%) of the total probability density within a discrete 3D volume.


**Reasoning**:
I will define the `find_enclosure_threshold` function and verify it using a 1s orbital generated with the existing `real_hydrogen_wavefunction` function.



In [ ]:
import numpy as np

def find_enclosure_threshold(prob_density, fraction=0.9):
    """
    Finds the probability density threshold that encloses a specific fraction
    of the total probability density in a discrete volume.
    """
    # 1. Flatten the 3D array
    flat_prob = prob_density.flatten()

    # 2. Sort in descending order
    sorted_prob = np.sort(flat_prob)[::-1]

    # 3. Calculate cumulative sum
    cumulative_prob = np.cumsum(sorted_prob)

    # 4. Total sum
    total_sum = np.sum(flat_prob)

    # 5. Find the index where cumulative sum reaches the target fraction
    target_val = fraction * total_sum
    threshold_idx = np.searchsorted(cumulative_prob, target_val)

    # 6. Return the threshold value
    return sorted_prob[threshold_idx]

# Verification with 1s orbital
extent = 5
res = 100
coords = np.linspace(-extent, extent, res)
X, Y, Z = np.meshgrid(coords, coords, coords)
r = np.sqrt(X**2 + Y**2 + Z**2)
theta = np.arccos(Z / (r + 1e-15))
phi = np.arctan2(Y, X)

# Calculate probability density for 1s orbital (n=1, l=0, m=0)
psi_1s = real_hydrogen_wavefunction(1, 0, 0, r, theta, phi)
prob_1s = psi_1s**2

# Find the 90% enclosure threshold
threshold_90 = find_enclosure_threshold(prob_1s, fraction=0.9)

print(f"1s Orbital 90% Enclosure Threshold: {threshold_90:.6e}")
print(f"Max Probability Density: {np.max(prob_1s):.6e}")
print(f"Threshold as % of Max: {(threshold_90 / np.max(prob_1s)) * 100:.2f}%")

## Update Isosurface Function

### Subtask:
Modify the 3D isosurface plotting functions to use the probability enclosure threshold logic instead of a fixed percentage of the maximum density.


**Reasoning**:
I will redefine the 3D isosurface plotting functions to incorporate the probability enclosure logic, replacing the fixed fraction threshold with the dynamic enclosure threshold calculated by the `find_enclosure_threshold` function.



In [ ]:
from skimage import measure
from matplotlib.ticker import MaxNLocator

def plot_orbital_isosurface(ax, n, l, m, label, extent=15, res=50, fraction=0.9):
    """
    Visualizes an orbital as a solid 3D isosurface that encloses a specific fraction
    of the total probability density.
    """
    # 1. Generate 3D Cartesian grid
    coords = np.linspace(-extent, extent, res)
    X, Y, Z = np.meshgrid(coords, coords, coords)

    # 2. Convert to spherical coordinates
    r = np.sqrt(X**2 + Y**2 + Z**2)
    theta = np.arccos(Z / (r + 1e-15))
    phi = np.arctan2(Y, X)

    # 3. Calculate probability density |psi|^2
    psi = real_hydrogen_wavefunction(n, l, m, r, theta, phi)
    prob = np.abs(psi)**2

    # 4. Determine isosurface level using enclosure threshold
    level = find_enclosure_threshold(prob, fraction=fraction)

    try:
        # 5. Generate surface mesh
        verts, faces, normals, values = measure.marching_cubes(prob, level)

        # 6. Map mesh indices back to coordinate space
        step = (2 * extent) / (res - 1)
        verts = verts * step - extent

        # 7. Render surface
        ax.plot_trisurf(verts[:, 0], verts[:, 1], faces, verts[:, 2],
                        cmap='Blues_r', lw=0, alpha=0.8, edgecolor='none')

        # 8. Standardize labels and title
        ax.set_xlabel('$x$', fontsize=24)
        ax.set_ylabel('$y$', fontsize=24)
        ax.set_zlabel('$z$', fontsize=24)
        ax.set_box_aspect([1, 1, 1])
        ax.set_xlim(-extent, extent)
        ax.set_ylim(-extent, extent)
        ax.set_zlim(-extent, extent)

    except ValueError:
        ax.text2D(0.5, 0.5, "No surface found", transform=ax.transAxes, ha='center')

def plot_real_isosurfaces_3x5(states_1s, states_2p, states_3d, res=50, fraction=0.9,
                               row_extents=[6, 11, 18]):
    """
    Visualizes 'real' hydrogen orbitals in a 3x5 grid using the 90% enclosure logic.
    """
    fig = plt.figure(figsize=(24, 18), facecolor='white')

    mapping = [
        (states_1s, [3], 0),
        (states_2p, [7, 8, 9], 1),
        (states_3d, [11, 12, 13, 14, 15], 2)
    ]

    for states, indices, row_idx in mapping:
        extent = row_extents[row_idx]
        for (n, l, m, label), idx in zip(states, indices):
            ax = fig.add_subplot(3, 5, idx, projection='3d')

            # Pass the fraction parameter to determine the enclosure level
            plot_orbital_isosurface(ax, n, l, m, label, extent=extent, res=res, fraction=fraction)

            # Refine styling for the 3x5 layout
            ax.tick_params(axis='both', which='major', labelsize=12)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.zaxis.set_major_locator(MaxNLocator(nbins=3))

            # Position the label underneath the plot
            ax.text2D(0.5, -0.15, rf'${label}$', transform=ax.transAxes,
                      ha='center', va='top', fontsize=32, fontweight='bold')

    plt.tight_layout(rect=[0, 0.05, 0.95, 0.95], h_pad=4.0, w_pad=3.0)
    plt.show()

print("Isosurface plotting functions updated with enclosure threshold logic.")

## Generate 90% Probability Grid

### Subtask:
Re-visualize the 3x5 grid of real hydrogen orbitals (1s, 2p, 3d) using isosurfaces that specifically enclose 90% of the total probability density.


**Reasoning**:
I will call the `plot_real_isosurfaces_3x5` function with the specified parameters to visualize the hydrogen orbitals using the 90% probability enclosure logic.



In [ ]:
# Execute the 3x5 isosurface grid visualization with 90% probability enclosure
# Extents updated to [6, 11, 18] to match the volumetric cloud (fuzzy) plots
plot_real_isosurfaces_3x5(
    real_row_1s,
    real_row_2p,
    real_row_3d,
    res=60,
    fraction=0.8,
    row_extents=[6, 11, 18]
)

In [ ]:
def plot_orbital_stochastic_dots(ax, n, l, m, label, extent=15, num_samples=200000, target_dots=15000):
    """
    Visualizes an orbital using stochastic point sampling (Monte Carlo).
    The density of points in space is proportional to |psi|^2.
    Dots are colored by the sign of the wavefunction (red: +, blue: -).
    """
    # 1. Generate random candidate points uniformly in the 3D volume
    X_rand = np.random.uniform(-extent, extent, num_samples)
    Y_rand = np.random.uniform(-extent, extent, num_samples)
    Z_rand = np.random.uniform(-extent, extent, num_samples)

    # 2. Convert to spherical coordinates for wavefunction calculation
    r = np.sqrt(X_rand**2 + Y_rand**2 + Z_rand**2)
    theta = np.arccos(Z_rand / (r + 1e-15))
    phi = np.arctan2(Y_rand, X_rand)

    # 3. Calculate real wavefunction and probability density
    psi = real_hydrogen_wavefunction(n, l, m, r, theta, phi)
    prob = np.abs(psi)**2

    # 4. Perform rejection sampling
    max_p = np.max(prob)
    acceptance_ratios = prob / max_p
    keep_mask = np.random.uniform(0, 1, num_samples) < acceptance_ratios

    # 5. Extract accepted points and their wavefunction values
    x_dots = X_rand[keep_mask]
    y_dots = Y_rand[keep_mask]
    z_dots = Z_rand[keep_mask]
    psi_dots = psi[keep_mask]

    # Limit to target_dots to keep plot performance smooth
    if len(x_dots) > target_dots:
        indices = np.random.choice(len(x_dots), target_dots, replace=False)
        x_dots, y_dots, z_dots = x_dots[indices], y_dots[indices], z_dots[indices]
        psi_dots = psi_dots[indices]

    # 6. Assign colors based on the sign of the wavefunction
    # Red for positive phase, Blue for negative phase
    colors = np.where(psi_dots > 0, 'red', 'blue')

    # 7. Render the dots
    ax.scatter(x_dots, y_dots, z_dots, s=5, facecolors=colors, alpha=0.3, edgecolors='none')

    # 8. Axis styling
    ax.set_xlabel('$x$', fontsize=24)
    ax.set_ylabel('$y$', fontsize=24)
    ax.set_zlabel('$z$', fontsize=24)
    ax.set_box_aspect([1, 1, 1])
    ax.set_xlim(-extent, extent)
    ax.set_ylim(-extent, extent)
    ax.set_zlim(-extent, extent)

def plot_real_stochastic_grid_3x5(states_1s, states_2p, states_3d, row_extents=[6, 11, 18]):
    """
    Generates the 3x5 grid using the Stochastic Dot Plot model with sign-based coloring.
    """
    fig = plt.figure(figsize=(24, 18), facecolor='white')

    mapping = [
        (states_1s, [3], 0),
        (states_2p, [7, 8, 9], 1),
        (states_3d, [11, 12, 13, 14, 15], 2)
    ]

    for states, indices, row_idx in mapping:
        extent = row_extents[row_idx]
        for (n, l, m, label), idx in zip(states, indices):
            ax = fig.add_subplot(3, 5, idx, projection='3d')

            # Use the updated stochastic dots function
            plot_orbital_stochastic_dots(ax, n, l, m, label, extent=extent)

            # Formatting
            ax.tick_params(axis='both', which='major', labelsize=12)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.zaxis.set_major_locator(MaxNLocator(nbins=3))

            # Position the label underneath the plot
            ax.text2D(0.5, -0.15, rf'${label}$', transform=ax.transAxes,
                      ha='center', va='top', fontsize=32, fontweight='bold')

    plt.tight_layout(rect=[0, 0.05, 0.95, 0.95], h_pad=4.0, w_pad=3.0)
    plt.show()

# Execute the 3x5 Stochastic Dot Grid with sign-based coloring
plot_real_stochastic_grid_3x5(real_row_1s, real_row_2p, real_row_3d)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

def plot_orbital_cloud_phase(ax, n, l, m, label, extent=15, res=55, threshold=0.005):
    """
    Visualizes an orbital as a 3D volumetric cloud with phase-based coloring (Red: +, Blue: -).
    """
    # 1. Generate 3D Cartesian grid
    coords = np.linspace(-extent, extent, res)
    X, Y, Z = np.meshgrid(coords, coords, coords)

    # 2. Convert to spherical coordinates
    r = np.sqrt(X**2 + Y**2 + Z**2)
    theta = np.arccos(Z / (r + 1e-15))
    phi = np.arctan2(Y, X)

    # 3. Compute real wavefunction and probability density
    psi = real_hydrogen_wavefunction(n, l, m, r, theta, phi)
    prob = np.abs(psi)**2

    # 4. Normalize and filter low-probability points
    max_p = np.max(prob)
    prob_norm = prob / max_p if max_p > 0 else prob
    mask = prob_norm > threshold

    x_f = X[mask].flatten()
    y_f = Y[mask].flatten()
    z_f = Z[mask].flatten()
    p_f = prob_norm[mask].flatten()
    psi_f = psi[mask].flatten()

    # 5. Map phase to color and density to transparency
    colors = np.zeros((len(p_f), 4))
    # Set RGB based on sign
    colors[psi_f > 0] = [1.0, 0.0, 0.0, 0.0]  # Red for positive phase
    colors[psi_f <= 0] = [0.0, 0.0, 1.0, 0.0] # Blue for negative phase

    # Set Alpha (transparency) based on density
    colors[:, -1] = np.clip(0.7 * p_f**0.4, 0, 1)

    # 6. Render scatter plot
    ax.scatter(x_f, y_f, z_f, facecolors=colors, s=12, edgecolors='none')

    # 7. Axis styling
    ax.set_xlabel('$x$', fontsize=24)
    ax.set_ylabel('$y$', fontsize=24)
    ax.set_zlabel('$z$', fontsize=24)
    ax.set_box_aspect([1, 1, 1])
    ax.set_xlim(-extent, extent)
    ax.set_ylim(-extent, extent)
    ax.set_zlim(-extent, extent)

def plot_real_phase_clouds_3x5(states_1s, states_2p, states_3d, res=50, threshold=0.005, row_extents=[6, 11, 18]):
    """
    Generates the 3x5 grid using the phase-colored volumetric cloud model.
    """
    fig = plt.figure(figsize=(24, 18), facecolor='white')

    mapping = [
        (states_1s, [3], 0),
        (states_2p, [7, 8, 9], 1),
        (states_3d, [11, 12, 13, 14, 15], 2)
    ]

    for states, indices, row_idx in mapping:
        extent = row_extents[row_idx]
        for (n, l, m, label), idx in zip(states, indices):
            ax = fig.add_subplot(3, 5, idx, projection='3d')
            plot_orbital_cloud_phase(ax, n, l, m, label, extent=extent, res=res, threshold=threshold)

            ax.tick_params(axis='both', which='major', labelsize=10)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.zaxis.set_major_locator(MaxNLocator(nbins=3))

            ax.text2D(0.5, -0.15, rf'${label}$', transform=ax.transAxes,
                      ha='center', va='top', fontsize=32, fontweight='bold')

    plt.tight_layout(rect=[0, 0.05, 0.95, 0.95], h_pad=4.0, w_pad=3.0)
    plt.show()

# Execute the phase-colored cloud visualization
plot_real_phase_clouds_3x5(real_row_1s, real_row_2p, real_row_3d)

# Task
Implement a function to calculate the probability density threshold for an 80% enclosure fraction. Develop a 3D plotting function that generates an isosurface at this threshold and splits the resulting mesh into two distinct components based on the sign of the wavefunction, rendering them in red (positive) and blue (negative). Generate a 3x5 grid of real hydrogen orbitals (1s, 2p, 3d) using this logic and summarize how the 80% enclosure and phase-coloring clarify the relationship between electron probability and wave symmetry.

## Implement Robust Enclosure Threshold

### Subtask:
Develop a function to calculate a probability density threshold that encloses 80% of the total probability, with safeguards for numerical stability.


**Reasoning**:
I will define the `calculate_enclosure_threshold` function with safeguards for numerical stability and error handling as requested, then verify it using a 2p_z hydrogen orbital.



In [ ]:
import numpy as np

def calculate_enclosure_threshold(prob_density, fraction=0.8):
    """
    Finds the probability density threshold that encloses a specific fraction
    of the total probability density in a discrete volume.
    Includes safeguards for numerical stability.
    """
    # Safeguard: Check if the array is empty or zeroed out
    total_sum = np.sum(prob_density)
    if total_sum <= 0:
        print("Warning: Probability density is zeroed out.")
        return 0.0

    # 1. Flatten the 3D array
    flat_prob = prob_density.flatten()

    # 2. Sort in descending order
    sorted_prob = np.sort(flat_prob)[::-1]

    # 3. Calculate cumulative sum
    cumulative_prob = np.cumsum(sorted_prob)

    # 4. Find the index where cumulative sum reaches the target fraction
    target_val = fraction * total_sum
    # Use searchsorted for efficiency
    threshold_idx = np.searchsorted(cumulative_prob, target_val)

    # Safeguard: Ensure the index is within bounds
    threshold_idx = min(threshold_idx, len(sorted_prob) - 1)

    # 5. Return the threshold value
    threshold = sorted_prob[threshold_idx]

    # Safeguard: Numerical stability for marching cubes
    # Avoid threshold being exactly 0 or exactly the maximum
    max_val = sorted_prob[0]
    min_val = sorted_prob[-1]

    # If the threshold is 0, set to a tiny non-zero value
    if threshold <= 1e-20:
        threshold = 1e-15

    # If threshold is too close to max, scale it down slightly
    if threshold >= max_val:
        threshold = 0.99 * max_val

    return threshold

# --- Test with 2p_z orbital (n=2, l=1, m=0) ---
extent_test = 15
res_test = 60
coords_test = np.linspace(-extent_test, extent_test, res_test)
X_t, Y_t, Z_t = np.meshgrid(coords_test, coords_test, coords_test)
r_t = np.sqrt(X_t**2 + Y_t**2 + Z_t**2)
theta_t = np.arccos(Z_t / (r_t + 1e-15))
phi_t = np.arctan2(Y_t, X_t)

# Calculate probability density for 2p_z orbital
psi_2pz = real_hydrogen_wavefunction(2, 1, 0, r_t, theta_t, phi_t)
prob_2pz = psi_2pz**2

# Calculate 80% enclosure threshold
th_80 = calculate_enclosure_threshold(prob_2pz, fraction=0.8)

print(f"--- 2p_z Orbital (80% Enclosure) ---")
print(f"Calculated Threshold: {th_80:.6e}")
print(f"Max Prob Density: {np.max(prob_2pz):.6e}")
print(f"Threshold as % of Max: {(th_80 / np.max(prob_2pz)) * 100:.2f}%")

## Implement Mesh-Splitting Isosurface Logic

### Subtask:
Develop a plotting function that generates a 3D isosurface and then splits the resulting mesh into two distinct components based on the sign of the wavefunction, rendering them in red (positive) and blue (negative).


**Reasoning**:
I will define the `plot_phase_isosurface` function as requested, implementing the logic to generate a 3D isosurface and split it into positive (red) and negative (blue) phase components based on the sign of the wavefunction.



In [ ]:
from skimage import measure

def plot_phase_isosurface(ax, n, l, m, label, extent=15, res=50, fraction=0.8):
    """
    Visualizes an orbital as a 3D isosurface split into positive (red)
    and negative (blue) components based on the wavefunction sign.
    """
    # 1. Generate 3D Cartesian grid
    coords = np.linspace(-extent, extent, res)
    X, Y, Z = np.meshgrid(coords, coords, coords)

    # 2. Convert to spherical coordinates
    r = np.sqrt(X**2 + Y**2 + Z**2)
    theta = np.arccos(Z / (r + 1e-15))
    phi = np.arctan2(Y, X)

    # 3. Compute real wavefunction and probability density
    psi = real_hydrogen_wavefunction(n, l, m, r, theta, phi)
    prob = psi**2

    # 4. Determine isosurface level for the enclosure fraction
    level = calculate_enclosure_threshold(prob, fraction=fraction)

    try:
        # 5. Extract isosurface mesh using marching cubes
        verts, faces, normals, values = measure.marching_cubes(prob, level)

        # 6. Map mesh indices back to physical coordinate space
        step = (2 * extent) / (res - 1)
        verts = verts * step - extent

        # 7. Identify vertex indices in the original grid to sample wavefunction phase
        # Marched vertices are floating point indices; we can sample phase at their nearest points or interpolate.
        # A common approach for marching_cubes result is to find phase by mapping vertices to the original 'psi' grid.

        # Re-map physical verts back to 'index space' for sampling psi
        v_idx = ((verts + extent) / step).astype(int)
        v_idx = np.clip(v_idx, 0, res - 1)

        # Sample wavefunction at vertex positions
        psi_at_verts = psi[v_idx[:, 0], v_idx[:, 1], v_idx[:, 2]]

        # 8. Separate faces into positive and negative components based on average vertex phase
        face_phases = np.mean(psi_at_verts[faces], axis=1)
        pos_faces = faces[face_phases > 0]
        neg_faces = faces[face_phases <= 0]

        # 9. Render the two components with distinct colors
        if len(pos_faces) > 0:
            ax.plot_trisurf(verts[:, 0], verts[:, 1], pos_faces, verts[:, 2],
                            color='red', lw=0, alpha=0.7, edgecolor='none', label='Positive Phase')

        if len(neg_faces) > 0:
            ax.plot_trisurf(verts[:, 0], verts[:, 1], neg_faces, verts[:, 2],
                            color='blue', lw=0, alpha=0.7, edgecolor='none', label='Negative Phase')

        # 10. Standardize axis appearance
        ax.set_xlabel('$x$', fontsize=24)
        ax.set_ylabel('$y$', fontsize=24)
        ax.set_zlabel('$z$', fontsize=24)
        ax.set_box_aspect([1, 1, 1])
        ax.set_xlim(-extent, extent)
        ax.set_ylim(-extent, extent)
        ax.set_zlim(-extent, extent)

    except (ValueError, RuntimeError) as e:
        ax.text2D(0.5, 0.5, f"Surface error: {e}", transform=ax.transAxes, ha='center')

print("Phase-splitting isosurface function 'plot_phase_isosurface' defined.")

## Generate 80% Phase-Colored Grid

### Subtask:
Create a 3x5 grid visualization of real hydrogen orbitals (1s, 2p, 3d) using the phase-splitting isosurface logic with an 80% enclosure fraction.


**Reasoning**:
I will define and execute the `plot_real_phase_isosurfaces_3x5` function to create the 3x5 grid of real hydrogen orbitals using the phase-splitting logic with an 80% enclosure threshold, following the specific subplot mapping and formatting instructions.



In [ ]:
from matplotlib.ticker import MaxNLocator

def plot_real_phase_isosurfaces_3x5(states_1s, states_2p, states_3d, res=60, fraction=0.8, row_extents=[6, 11, 18]):
    """
    Generates a 3x5 grid of 3D isosurfaces split by phase (Red: +, Blue: -)
    using a specific enclosure fraction of the probability density.
    """
    fig = plt.figure(figsize=(24, 18), facecolor='white')

    # Subplot mapping: row index, subplot index, and extent index
    mapping = [
        (states_1s, [3], 0),                # Row 1 (1s)
        (states_2p, [7, 8, 9], 1),          # Row 2 (2p)
        (states_3d, [11, 12, 13, 14, 15], 2) # Row 3 (3d)
    ]

    for states, indices, row_idx in mapping:
        extent = row_extents[row_idx]
        for (n, l, m, label), idx in zip(states, indices):
            ax = fig.add_subplot(3, 5, idx, projection='3d')

            # Call the phase-splitting isosurface function
            plot_phase_isosurface(ax, n, l, m, label, extent=extent, res=res, fraction=fraction)

            # Formatting: remove title set by internal function to move label underneath
            ax.set_title('')

            # Adjust ticks and labels
            ax.tick_params(axis='both', which='major', labelsize=12)
            ax.xaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.yaxis.set_major_locator(MaxNLocator(nbins=3))
            ax.zaxis.set_major_locator(MaxNLocator(nbins=3))

            # Add bold orbital label underneath
            ax.text2D(0.5, -0.15, rf'${label}$', transform=ax.transAxes,
                      ha='center', va='top', fontsize=32, fontweight='bold')

    plt.tight_layout(rect=[0, 0.05, 0.95, 0.95], h_pad=4.0, w_pad=3.0)
    plt.show()

# Execute the 3x5 Phase-Colored Isosurface Grid with 80% enclosure
plot_real_phase_isosurfaces_3x5(real_row_1s, real_row_2p, real_row_3d, res=60, fraction=0.8)